In [ ]:
# Databricks notebook source
# MAGIC %md
# MAGIC # 🧹 ETL TEST UTILITY — TRUNCATE + SEED DATA
# MAGIC **Purpose:** Reset all pipeline tables and load structured test data.
# MAGIC **Run this BEFORE the main ETL pipeline during testing.**
# MAGIC **DO NOT run this in production.**

# COMMAND ----------

import pyspark.sql.functions as F
from pyspark.sql.types import *
from datetime import datetime, date, timedelta

DATABASE = "dbo"

def tbl(name):
    return f"{DATABASE}.{name}"

ALL_TABLES = [
    "person", "bronze_person", "silver_person", "gold_person",
    "dim_person", "quarantine_person", "etl_control", "audit_trail",
    "rca_errors", "dim_late_arrival_log", "column_lineage"
]

# COMMAND ----------

# MAGIC %md
# MAGIC ## STEP 1 — TRUNCATE ALL TABLES

# COMMAND ----------

def truncate_all_tables():
    print("=" * 60)
    print("🗑️  TRUNCATING ALL PIPELINE TABLES")
    print("=" * 60)
    for t in ALL_TABLES:
        full = tbl(t)
        try:
            spark.sql(f"DROP TABLE IF EXISTS {full}")
            print(f"  ✅ Dropped : {full}")
        except Exception as e:
            print(f"  ⚠️  Failed  : {full} — {e}")
    print("=" * 60)
    print("✅ All tables cleared. Ready for fresh seed.")
    print("=" * 60)

truncate_all_tables()

# COMMAND ----------

# MAGIC %md
# MAGIC ## STEP 2 — SEED TEST DATA
# MAGIC
# MAGIC | Batch | Records | person_ids | Purpose |
# MAGIC |-------|---------|------------|---------|
# MAGIC | 1 | 15 | 1001–1015 | Clean baseline — all pass DQ |
# MAGIC | 2 | 3  | 9991–9993 | DQ failures — NULL id / bad gender / future DOB |
# MAGIC | 3 | 5  | 1001,1003,1005,1007,1009 | Late arrivals — business_date Jan-10, arrive Feb-28, gender changed |
# MAGIC
# MAGIC **Run order:**
# MAGIC 1. Load Batch 1 + 2 into source → run ETL → verify baseline
# MAGIC 2. Load Batch 3 into source → run ETL → verify late-arrival stitching

# COMMAND ----------

def seed_test_data():
    """
    Seed the source person table with exactly the real schema:
      location_id / provider_id / care_site_id  → STRING
      birth_datetime                             → DATE
      business_effective_date                    → DATE  (key late-arrival field)
    """

    # Schema exactly matching dbo.person
    schema = StructType([
        StructField("person_id",                    IntegerType(),   True),
        StructField("gender_concept_id",             IntegerType(),   True),
        StructField("year_of_birth",                 IntegerType(),   True),
        StructField("month_of_birth",                IntegerType(),   True),
        StructField("day_of_birth",                  IntegerType(),   True),
        StructField("birth_datetime",                DateType(),      True),  # DATE
        StructField("race_concept_id",               IntegerType(),   True),
        StructField("ethnicity_concept_id",          IntegerType(),   True),
        StructField("location_id",                   StringType(),    True),  # STRING
        StructField("provider_id",                   StringType(),    True),  # STRING
        StructField("care_site_id",                  StringType(),    True),  # STRING
        StructField("person_source_value",           StringType(),    True),
        StructField("gender_source_value",           StringType(),    True),
        StructField("gender_source_concept_id",      IntegerType(),   True),
        StructField("race_source_value",             StringType(),    True),
        StructField("race_source_concept_id",        IntegerType(),   True),
        StructField("ethnicity_source_value",        StringType(),    True),
        StructField("ethnicity_source_concept_id",   IntegerType(),   True),
        StructField("created_timestamp",             TimestampType(), True),
        StructField("updated_timestamp",             TimestampType(), True),
        StructField("is_deleted",                    BooleanType(),   True),
        StructField("business_effective_date",       DateType(),      True),  # KEY FIELD
    ])

    # Gender codes
    MALE    = 8507
    FEMALE  = 8532
    UNKNOWN = 8551

    def rec(pid, gender_id, yob, mob, dob, biz_date, upd_ts, cre_ts=None):
        """Build one source record. cre_ts defaults to upd_ts."""
        if cre_ts is None:
            cre_ts = upd_ts
        gsv = {MALE: "M", FEMALE: "F", UNKNOWN: "U"}.get(gender_id, "U")
        return (
            pid,                                    # person_id
            gender_id,                              # gender_concept_id
            yob,                                    # year_of_birth
            mob,                                    # month_of_birth
            dob,                                    # day_of_birth
            date(yob, mob, dob),                   # birth_datetime (DATE)
            8527,                                   # race_concept_id
            38003564,                               # ethnicity_concept_id
            str(pid * 10),                          # location_id  (STRING)
            "1001",                                 # provider_id  (STRING)
            "2001",                                 # care_site_id (STRING)
            f"PSV_{pid:04d}",                       # person_source_value
            gsv,                                    # gender_source_value
            gender_id,                              # gender_source_concept_id
            "WHITE",                                # race_source_value
            8527,                                   # race_source_concept_id
            "NON-HISPANIC",                         # ethnicity_source_value
            38003564,                               # ethnicity_source_concept_id
            cre_ts,                                 # created_timestamp
            upd_ts,                                 # updated_timestamp
            False,                                  # is_deleted
            biz_date,                               # business_effective_date (DATE)
        )

    # ── BATCH 1: 15 clean baseline records ───────────────────────────────
    # Arrival date = business date = 03 Jan 2026
    b1_biz = date(2026, 1, 3)
    b1_ts  = datetime(2026, 1, 3, 9, 0, 0)

    genders_b1 = [MALE, FEMALE, MALE, FEMALE, MALE,
                  FEMALE, MALE, FEMALE, MALE, FEMALE,
                  MALE, FEMALE, MALE, FEMALE, MALE]

    batch1 = []
    for i in range(15):
        pid = 1001 + i
        mob = 1 + (i % 12)
        dob = 1 + (i % 28)
        yob = 1970 + i
        ts  = datetime(2026, 1, 3, 9, 0, i)       # unique per record
        batch1.append(rec(pid, genders_b1[i], yob, mob, dob,
                          b1_biz, ts))

    # ── BATCH 2: 3 DQ-failure records ────────────────────────────────────
    b2_biz = date(2026, 1, 30)
    b2_ts  = datetime(2026, 1, 30, 9, 0, 0)

    batch2 = [
        # NULL person_id → NOT_NULL_person_id rule fails
        (None, MALE,   1990, 5, 15, date(1990,5,15), 8527, 38003564,
         "99910", "1001", "2001", "PSV_9991", "M", MALE,
         "WHITE", 8527, "NON-HISPANIC", 38003564,
         b2_ts, b2_ts, False, b2_biz),

        # Invalid gender 9999 → VALID_GENDER_CONCEPT rule fails
        rec(9992, 9999, 1975, 3, 22, b2_biz, datetime(2026, 1, 30, 9, 1, 0)),

        # Future year of birth 2099 → VALID_YEAR_OF_BIRTH rule fails
        (9993, FEMALE, 2099, 1, 1, date(2099,1,1), 8527, 38003564,
         "99930", "1001", "2001", "PSV_9993", "F", FEMALE,
         "WHITE", 8527, "NON-HISPANIC", 38003564,
         datetime(2026,1,30,9,2,0), datetime(2026,1,30,9,2,0), False, b2_biz),
    ]

    # ── BATCH 3: 5 late-arrival records ──────────────────────────────────
    # business_effective_date = 10 Jan 2026 (BEFORE the batch-1 watermark)
    # updated_timestamp       = 28 Feb 2026 (system arrival — AFTER watermark)
    # Gender is changed so SCD Type-2 fires
    b3_biz = date(2026, 1, 10)           # ← business date (10-Jan)
    b3_arr = datetime(2026, 2, 28, 9, 0, 0)  # ← system arrival (28-Feb)

    late_pids    = [1001, 1003, 1005, 1007, 1009]
    late_genders = [UNKNOWN, UNKNOWN, FEMALE, UNKNOWN, FEMALE]   # all differ from batch1

    batch3 = []
    for idx, (pid, gid) in enumerate(zip(late_pids, late_genders)):
        i   = pid - 1001
        mob = 1 + (i % 12)
        dob = 1 + (i % 28)
        yob = 1970 + i
        arr_ts = datetime(2026, 2, 28, 9, 0, idx)
        batch3.append(rec(pid, gid, yob, mob, dob,
                          b3_biz,          # business_effective_date = 10-Jan
                          arr_ts,          # updated_timestamp       = 28-Feb (arrival)
                          arr_ts))         # created_timestamp        = 28-Feb

    all_rows = batch1 + batch2 + batch3

    df = spark.createDataFrame(all_rows, schema=schema)
    df.write.format("delta").mode("overwrite").saveAsTable(tbl("person"))

    # ── Print summaries ───────────────────────────────────────────────────
    print("=" * 70)
    print("🧪 TEST DATA SEEDED INTO dbo.person")
    print("=" * 70)
    print(f"  Batch 1 (clean baseline)        : 15 records  — person_id 1001–1015")
    print(f"    business_effective_date        : 2026-01-03")
    print(f"    Genders                        : Male / Female alternating")
    print(f"")
    print(f"  Batch 2 (DQ failures)           :  3 records  — person_id 9991–9993")
    print(f"    9991 — NULL person_id          : fails NOT_NULL_person_id")
    print(f"    9992 — gender_concept_id=9999  : fails VALID_GENDER_CONCEPT")
    print(f"    9993 — year_of_birth=2099      : fails VALID_YEAR_OF_BIRTH")
    print(f"")
    print(f"  Batch 3 (late arrivals)         :  5 records  — person_id 1001,1003,1005,1007,1009")
    print(f"    business_effective_date        : 2026-01-10  ← BEFORE batch-1 watermark")
    print(f"    updated_timestamp (arrival)    : 2026-02-28  ← AFTER watermark")
    print(f"    Gender changed to Unknown/Female → SCD Type-2 must fire")
    print("=" * 70)

    print("\n📋 All source rows (sorted by business_effective_date, person_id):")
    spark.table(tbl("person")) \
        .select("person_id", "gender_concept_id", "year_of_birth",
                "business_effective_date", "updated_timestamp") \
        .orderBy("business_effective_date", "person_id") \
        .show(25, truncate=False)

    print("=" * 70)
    print("✅ Seed complete.")
    print("   ► Run the ETL pipeline (Notebook B) with Batch 1+2 first.")
    print("   ► Then add Batch 3 rows (already in table) and re-run to test late arrival.")
    print("=" * 70)

seed_test_data()

# COMMAND ----------

# MAGIC %md
# MAGIC ## STEP 3 — VERIFY SOURCE TABLE

# COMMAND ----------

print("Source table record count:", spark.table(tbl("person")).count())
spark.table(tbl("person")).printSchema()
